In [8]:
!pip install prophet -q

In [9]:
import pandas as pd
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")
print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [10]:
df = pd.read_csv("cleaned_superstore_data.xlsx - Sheet1.csv")
print(f"✅ Loaded {len(df):,} rows")
print("Columns:", df.columns.tolist())

✅ Loaded 9,994 rows
Columns: ['row id', 'order id', 'Order Date', 'ship date', 'ship mode', 'customer id', 'customer name', 'segment', 'postal code', 'city', 'state', 'country', 'region_x', 'market', 'product id', 'product name', 'sub-category', 'category', 'Sales', 'quantity', 'discount', 'profit', 'shipping cost', 'order priority', 'returned', 'region_y', 'person', 'region']


In [11]:
# Convert Order Date to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])

# Aggregate to monthly sales (this creates monthly_sales)
monthly_sales = (
    df.groupby(pd.Grouper(key="Order Date", freq="MS"))["Sales"]
    .sum()
    .reset_index()
)

# Rename to what Prophet needs
monthly_sales = monthly_sales.rename(columns={"Order Date": "ds", "Sales": "y"})

print(f"✅ Preprocessed to {len(monthly_sales)} monthly data points")
monthly_sales.head()

✅ Preprocessed to 48 monthly data points


,ds,y
0,2014-01-01,13946.229
1,2014-02-01,7013.709
2,2014-03-01,53607.746
3,2014-04-01,28175.457
4,2014-05-01,28836.807


In [12]:
model = Prophet(yearly_seasonality=True)
model.fit(monthly_sales)

print("✅ Prophet model trained successfully")

13:47:48 - cmdstanpy - INFO - Chain [1] start processing
13:47:48 - cmdstanpy - INFO - Chain [1] done processing


✅ Prophet model trained successfully


In [13]:
future = model.make_future_dataframe(periods=12, freq="MS")
forecast = model.predict(future)

# Prepare final output
results = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
results = results.merge(monthly_sales[["ds", "y"]], on="ds", how="left")

# Save the file that app.py needs
results.to_csv("forecast_results.csv", index=False)

print("✅ Forecast saved as forecast_results.csv")
print(f"   Historical months : {results['y'].notna().sum()}")
print(f"   Forecast months   : 12")
results.tail(15)   # show last few rows

✅ Forecast saved as forecast_results.csv
   Historical months : 48
   Forecast months   : 12


,ds,yhat,yhat_lower,yhat_upper,y
45,2017-10-01,66706.331478,59338.313146,73762.850212,77793.7552
46,2017-11-01,100555.767528,92855.652230,107172.719313,112326.4710
47,2017-12-01,96946.840165,89991.767090,104111.875278,90474.6008
48,2018-01-01,46886.318054,39269.177828,53687.498315,NaN
49,2018-02-01,34728.700024,27242.841898,42019.849995,NaN
50,2018-03-01,71672.049293,64699.168916,78361.957508,NaN
51,2018-04-01,58642.500846,51803.246569,65964.097273,NaN
52,2018-05-01,60430.657883,52998.650379,67170.830709,NaN
53,2018-06-01,57875.231352,50552.116265,65158.760718,NaN
54,2018-07-01,61009.382807,53320.519736,67757.157197,NaN
